In [ ]:
import torch as th
from torch import nn
from torch.nn import functional as F
from vesuvius_challenge.networks.transformer import WindowedTransformer

In [ ]:
SIZES = (8, 8, 8)
BATCH_SIZE = 4

In [ ]:
x = th.randn(BATCH_SIZE, 1, *SIZES).cuda()
y = th.randint(0, 1, (BATCH_SIZE, 1, *SIZES)).to(th.float).cuda()

In [ ]:
conv = nn.Conv3d(1, 8, 3, 1, 1).cuda()
trf = WindowedTransformer(8, 16, 3, 1, num_heads=4).cuda()



In [ ]:
out_x_conv = conv(x)
out_y_conv = conv(y)

out_trf = trf(out_x_conv, out_y_conv)

In [ ]:
out_trf.size()

In [ ]:
with th.no_grad():
    out_gen = trf(out_x_conv)
    print(out_gen.size())

# Exploration

In [ ]:
class WindowedTransformer(nn.Module):
    def __init__(
            self,
            channels: int,
            hidden: int,
            kernel_size: int,
            padding: int,
            num_heads: int = 8,
            encoder_layers: int = 3,
            decoder_layers: int = 3
    ) -> None:
        super().__init__()
        
        self.__channels = channels
        self.__kernel_size = kernel_size
        self.__padding = padding
        
        self.__trf = nn.Transformer(
            channels, nhead=num_heads, num_encoder_layers=encoder_layers, num_decoder_layers=decoder_layers, dim_feedforward=hidden, batch_first=True
        )
    
    def forward(self, x: th.Tensor) -> th.Tensor:
        assert len(x.size()) == 4
        
        b, _, w, h = x.size()
        
        device = "cuda" if next(self.parameters()).is_cuda else "cpu"
        
        input_trf = (
            F.unfold(
                x,
                self.__kernel_size,
                dilation=1,
                padding=self.__padding,
                stride=1,
            )
            .view(b, self.__channels, self.__kernel_size**2, -1)
            .permute(0, 3, 2, 1)
            .view(-1, self.__kernel_size**2, self.__channels)
        )
        
        input_trf = th.cat(
            [
                th.zeros(input_trf.size(0), 1, self.__channels, device=device),
                input_trf
            ],
            dim=1,
        )
        
        out = th.zeros(input_trf.size(0), 1, self.__channels, device=device)
        
        for _ in range(self.__kernel_size**2):
            out = th.cat([out, self.__trf(input_trf, out)[:, -1, None, :]], dim=1)
        
        out = (
            out
            .view(b, -1, self.__kernel_size**2 + 1, self.__channels)[:, :, 1:, :]
            .permute(0, 3, 2, 1)
            .contiguous()
            .view(b, self.__channels * self.__kernel_size ** 2, -1)
        )
        
        
        out = F.fold(out, (w, h), self.__kernel_size, dilation=1, padding=self.__padding, stride=1)
        
        return out


In [ ]:
conv_2d = nn.Conv2d(1, 8, (3, 3), (1, 1), (1, 1))

In [ ]:
out_conv = conv_2d(x)

In [ ]:
win_trf = WindowedTransformer(
    8, 16, 3, 1, 4, 3, 3
)
win_trf.cuda()

In [ ]:
with th.no_grad():
    out = win_trf(out_conv.cuda())

In [ ]:
out.size()

In [ ]:
trf = nn.Transformer(
    d_model=8,
    nhead=4,
    num_encoder_layers=3,
    num_decoder_layers=3,
    dim_feedforward=16,
    batch_first=True,
)

In [ ]:
unfold = nn.Unfold((3, 3), dilation=(1, 1), padding=(1, 1), stride=(1, 1))

In [ ]:
out_fold = unfold(out_conv)
b, c, w, h = out_conv.size()
out_fold = out_fold.view(b, c, 3 * 3, w * h)

In [ ]:
out_fold.size()

In [ ]:
nb_batch, nb_patch = out_fold.size(0), out_fold.size(-1)

In [ ]:
input_trf = out_fold.permute(0, 3, 2, 1).view(nb_batch * nb_patch, -1, c)

In [ ]:
input_trf.size()

In [ ]:
nb_batch * nb_patch, c -1

In [ ]:
tgt_trf = (
    unfold(conv_2d(y))
    .view(nb_batch, c, 3 * 3, nb_patch)
    .permute(0, 3, 2, 1)
    .view(nb_batch * nb_patch, -1, c)
)

In [ ]:
tgt_trf.size()

In [ ]:
with th.no_grad():
    out = trf(input_trf, tgt_trf)
    out = (
        out
        .view(nb_batch, nb_patch, 3 * 3, c)
        .permute(0, 3, 2, 1)
        .contiguous()
        .view(nb_batch, c * 3 * 3, nb_patch)
    )

In [ ]:
out.size()

In [ ]:
fold = nn.Fold((w, h), 3, 1, 1, 1)

In [ ]:
out_folded = fold(out)

In [ ]:
out_folded.size()